# Render the Apple on-device LLM explainer in 1080p

This notebook clones the **public** repo and renders **1920×1080, 30 fps, H.264 CRF 18, AAC 320 kbps**.

- No GitHub token is needed.
- 1080p keeps the file size manageable — much smaller than the 4K render.
- Rendering is CPU-bound (headless Chrome + x264). A free Colab CPU runtime may take **1–3 hours** for the full 12-minute video. Colab Pro / High-RAM is faster.
- The renderer prints every completed frame and percentage in real time.
- Do not close the tab while the full render is running.

In [ ]:
%%bash
set -e
apt-get -qq update
apt-get -qq install -y ffmpeg git curl
curl -fsSL https://deb.nodesource.com/setup_20.x | bash -
apt-get -qq install -y nodejs
node -v
npm -v

In [ ]:
%%bash
apt-get -qq install -y libnss3 libatk1.0-0 libatk-bridge2.0-0 libcups2 libdrm2 libxkbcommon0 libxcomposite1 libxdamage1 libxfixes3 libxrandr2 libgbm1 libasound2 libpango-1.0-0 libcairo2 || true

In [ ]:
import os, shutil, subprocess

REPO = 'lroe/How-Apple-Fits-a-Large-Language-Model-on-Your-iPhone.'

if os.path.exists('/content/apple'):
    shutil.rmtree('/content/apple')

url = f'https://github.com/{REPO}.git'
subprocess.run(['git', 'clone', '--depth', '1', url, '/content/apple'], check=True)
%cd /content/apple/video

In [ ]:
!npm install --no-audit --no-fund
!npx remotion browser ensure

## 1. Quick 1080p test (12 frames)
Validates the browser, audio and render path before the long run.

In [ ]:
!CONCURRENCY=4 node render.mjs --test

## 2. Full 1080p render
Every frame is printed as it completes. This is the long step.

If Colab gives you more CPUs, try `CONCURRENCY=8`.

In [ ]:
!CONCURRENCY=4 node render.mjs

In [ ]:
!ls -lh out/apple-explainer-1080p.mp4
!ffprobe -v error -select_streams v:0 -show_entries stream=width,height,r_frame_rate,codec_name,bit_rate -of default=nw=1 out/apple-explainer-1080p.mp4
!ffprobe -v error -select_streams a:0 -show_entries stream=codec_name,bit_rate -of default=nw=1 out/apple-explainer-1080p.mp4

## 3. Save to Google Drive
Drive is more reliable than a browser download for a long video.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/apple-render
!cp /content/apple/video/out/apple-explainer-1080p.mp4 /content/drive/MyDrive/apple-render/
!ls -lh /content/drive/MyDrive/apple-render/

## YouTube upload note
Upload the 1080p file as-is. Keep the 320 kbps AAC audio — do not re-encode it in an editor before uploading. YouTube will take a few minutes to finish HD processing after the upload.

Need 4K later? Run `!CONCURRENCY=4 node render.mjs --4k` instead (larger file, slower render).